<a href="https://colab.research.google.com/github/DJChallas/group1/blob/main/Group1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load the illinois.csv file into a pandas DataFrame
df = pd.read_csv('illinois.csv')

# Display the first 5 rows of the DataFrame
print(df.head())

   Area Number                       Industry Title Series Code     Jan-90  \
0            0                  Total Nonfarm         00-000000  5,270,100   
1            0                               Mining   10-000000     18,800   
2            0                   Construction         20-000000    230,700   
3            0                   Manufacturing        30-000000    923,700   
4            0   Trade, Transportation, & Utilities   40-000000  1,130,700   

      Feb-90     Mar-90     Apr-90     May-90     Jun-90     Jul-90  ...  \
0  5,279,100  5,291,300  5,282,100  5,289,600  5,299,200  5,303,300  ...   
1     18,800     18,600     18,300     18,100     18,300     18,300  ...   
2    230,500    229,400    227,000    224,600    223,000    224,400  ...   
3    927,300    927,200    923,200    924,100    922,600    918,100  ...   
4  1,131,600  1,132,600  1,131,600  1,131,600  1,133,900  1,131,700  ...   

      Nov-25     Dec-25     Jan-26     Feb-26     Mar-26     Apr-26  \
0  

### Illinois Industry Employment Trends (Interactive Plotly Visualization)

In [ ]:
import plotly.express as px

# 1. Illinois Industry Employment Trends (Muted Color Scheme)
fig_trends_muted = px.line(
    melted_df,
    x='Month',
    y='Employment',
    color='Industry Title',
    title='Illinois Industry Employment Trends (Feb 1990 - Aug 2026)',
    labels={'Month': 'Date', 'Employment': 'Employment'},
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig_trends_muted.update_layout(
    hovermode="x unified",
    legend_title_text='Industry',
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
    paper_bgcolor='white',
    plot_bgcolor='white',
    title_x=0.5
)

fig_trends_muted.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')
fig_trends_muted.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')

fig_trends_muted.show()

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

# Fallback check: If recession_dips_df_strict is not in memory, calculate it on the fly
if 'recession_dips_df_strict' in globals():
    dips_df = globals()['recession_dips_df_strict']
else:
    # Identify 5th percentile strict dips using melted_df_sorted
    negative_changes_per_industry_strict = melted_df_sorted[melted_df_sorted['Employment_Change'] < 0]
    dip_thresholds_strict = negative_changes_per_industry_strict.groupby('Industry Title')['Employment_Change'].quantile(0.05)

    significant_dips_list_strict = []
    for industry, threshold in dip_thresholds_strict.items():
        industry_dips_strict = melted_df_sorted[
            (melted_df_sorted['Industry Title'] == industry) &
            (melted_df_sorted['Employment_Change'] <= threshold)
        ].copy()
        significant_dips_list_strict.append(industry_dips_strict)

    if significant_dips_list_strict:
        dips_df = pd.concat(significant_dips_list_strict)
    else:
        dips_df = pd.DataFrame(columns=['Industry Title', 'Month', 'Employment', 'Employment_Change'])

    dips_df = dips_df[['Industry Title', 'Month', 'Employment_Change']]
    dips_df.rename(columns={'Employment_Change': 'Dip_Magnitude'}, inplace=True)
    dips_df['Abs_Dip_Magnitude'] = dips_df['Dip_Magnitude'].abs()
    # Save it globally so it's defined moving forward
    globals()['recession_dips_df_strict'] = dips_df

# Create the base line plot for overall employment trends using the Safe color sequence
fig_combined = px.line(
    melted_df,
    x='Month',
    y='Employment',
    color='Industry Title',
    title='Illinois Industry Employment Trends with 5th Percentile Dips Highlighted',
    labels={'Month': 'Date', 'Employment': 'Employment'},
    color_discrete_sequence=px.colors.qualitative.Safe
)

# Filter to only include industries present in melted_df
dips_to_overlay = dips_df[dips_df['Industry Title'].isin(melted_df['Industry Title'].unique())]

# Loop through each industry to add scatter markers for dips
for industry in dips_to_overlay['Industry Title'].unique():
    industry_dips = dips_to_overlay[dips_to_overlay['Industry Title'] == industry]
    # Get the corresponding employment value from melted_df for each dip month
    employment_at_dip = []
    for _, row in industry_dips.iterrows():
        matching_employment = melted_df[(melted_df['Industry Title'] == industry) & (melted_df['Month'] == row['Month'])]['Employment']
        if not matching_employment.empty:
            employment_at_dip.append(matching_employment.iloc[0])
        else:
            employment_at_dip.append(None)

    fig_combined.add_trace(
        go.Scatter(
            x=industry_dips['Month'],
            y=employment_at_dip,
            mode='markers',
            marker=dict(symbol='triangle-down', size=10, color='red', line=dict(width=1, color='DarkRed')),
            name=f'{industry} Dip',
            hoverinfo='text',
            hovertext=industry_dips.apply(lambda row: f"Industry: {row['Industry Title']}<br>Date: {row['Month'].strftime('%Y-%m')}<br>Dip Magnitude: {row['Dip_Magnitude']:,}", axis=1),
            showlegend=False
        )
    )

# Update layout for better readability
fig_combined.update_layout(
    hovermode="x unified",
    legend_title_text='Industry',
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02
    ),
    paper_bgcolor='white',
    plot_bgcolor='white',
    title_x=0.5
)

# Add grid for readability
fig_combined.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')
fig_combined.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')

fig_combined.show()

In [257]:
import pandas as pd
import re
import numpy as np
import plotly.express as px
import os

# 1. Direct path lookup for the dataset
csv_paths = ['illinois.csv', '../illinois.csv', '/content/illinois.csv', '/illinois.csv']
df_local = None

for path in csv_paths:
    if os.path.exists(path):
        df_local = pd.read_csv(path)
        break

if df_local is None:
    if 'df' in globals():
        df_local = globals()['df'].copy()
    else:
        raise FileNotFoundError("Could not locate 'illinois.csv' file or loaded 'df' variable in memory. Please run the first cell to load the data.")

# 2. Process and filter
df_local.columns = df_local.columns.str.strip()
industries_df = df_local[df_local['Industry Title'].str.strip() != 'Total Nonfarm'].copy()

month_year_pattern = re.compile(r'^[A-Za-z]{3}-\d{2}$', re.IGNORECASE)
potential_month_cols = [col for col in industries_df.columns if month_year_pattern.match(col)]
try:
    start_col_index = potential_month_cols.index('Feb-90')
    selected_month_cols = potential_month_cols[start_col_index:]
except ValueError:
    selected_month_cols = potential_month_cols

plot_df = industries_df[['Industry Title'] + selected_month_cols]
melted_df = plot_df.melt(id_vars=['Industry Title'], var_name='Month', value_name='Employment')
melted_df['Employment'] = melted_df['Employment'].astype(str).str.replace(',', '', regex=False)
melted_df['Employment'] = pd.to_numeric(melted_df['Employment'], errors='coerce')
melted_df['Month'] = pd.to_datetime(melted_df['Month'], format='%b-%y')
melted_df.dropna(subset=['Employment'], inplace=True)

melted_df_sorted = melted_df.sort_values(by=['Industry Title', 'Month'])
melted_df_sorted['Employment_Change'] = melted_df_sorted.groupby('Industry Title')['Employment'].diff()

negative_changes = melted_df_sorted[melted_df_sorted['Employment_Change'] < 0]

# Stating and confirming the use of the 2nd percentile threshold
dip_thresholds_very_strict = negative_changes.groupby('Industry Title')['Employment_Change'].quantile(0.02)

significant_dips_list_very_strict = []
for industry, threshold in dip_thresholds_very_strict.items():
    industry_dips_very_strict = melted_df_sorted[
        (melted_df_sorted['Industry Title'] == industry) &
        (melted_df_sorted['Employment_Change'] <= threshold)
    ].copy()
    significant_dips_list_very_strict.append(industry_dips_very_strict)

recession_dips_df_very_strict = pd.concat(significant_dips_list_very_strict)
recession_dips_df_very_strict.rename(columns={'Employment_Change': 'Dip_Magnitude'}, inplace=True)
recession_dips_df_very_strict['Abs_Dip_Magnitude'] = recession_dips_df_very_strict['Dip_Magnitude'].abs()

# 3. Handle symmetric log transformation manually for clean plotting of negative values
recession_dips_df_very_strict['Log_Dip_Coordinate'] = -np.log10(recession_dips_df_very_strict['Abs_Dip_Magnitude'])
max_dip = recession_dips_df_very_strict['Abs_Dip_Magnitude'].max()

ticks = [10**i for i in range(1, int(np.ceil(np.log10(max_dip))) + 1)]
ticks = [t for t in ticks if t <= max_dip]
tick_positions = [-np.log10(t) for t in ticks]
tick_labels = [f'-{t:,}' for t in ticks]

# 4. Interactive Plot using Safe muted layout styling (markers + muted dashed lines)
fig_downward_log = px.line(
    recession_dips_df_very_strict,
    x='Month',
    y='Log_Dip_Coordinate',
    color='Industry Title',
    title='Downward Visualized Monthly Employment Dips - 2nd Percentile (Symmetric Log Scale)',
    labels={
        'Month': 'Date',
        'Log_Dip_Coordinate': 'Contraction Severity (2nd Percentile Dips)'
    },
    color_discrete_sequence=px.colors.qualitative.Safe,
    hover_data={'Month': '|%Y-%m-%d', 'Industry Title': True, 'Dip_Magnitude': ':, .0f', 'Log_Dip_Coordinate': False}
)

# Update traces to include markers and muted dashed lines
fig_downward_log.update_traces(
    mode='lines+markers',
    line=dict(dash='dash', width=1.5),
    marker=dict(size=8, opacity=0.8, line=dict(width=1, color='DimGray'))
)

fig_downward_log.update_layout(
    title_x=0.5,
    hovermode="x unified",
    legend_title_text='Industry',
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
    paper_bgcolor='white',
    plot_bgcolor='white'
)

fig_downward_log.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')
fig_downward_log.update_yaxes(
    showgrid=True,
    gridwidth=1,
    gridcolor='#E8E8E8',
    tickvals=tick_positions,
    ticktext=tick_labels,
    title_text='Monthly Dip Magnitude (2nd Percentile, Log Scale)'
)

fig_downward_log.show()

### Methodology & Parameters for Very Extreme 2nd Percentile Dips

To map and compare the severe contraction events across different sectors, this visualization uses the following structured quantitative approach:

1. **Time Frame**:
   - The analysis spans the full duration of our historical dataset: **February 1990 to August 2026**.

2. **Calculation Method**:
   - **Month-over-Month (MoM) Changes**: First, the monthly employment change is calculated for each industry title:
     $$\Delta E_{i, t} = Employment_{i, t} - Employment_{i, t-1}$$
   - **Isolating Contractions**: The dataset is filtered to only include negative monthly changes (where $\Delta E_{i, t} < 0$).
   - **Threshold Determination (2nd Percentile)**: For each industry, we compute a customized "severe threshold" equivalent to its **2nd percentile (0.02 quantile)** of negative changes. This isolates only the absolute worst 2% of monthly job contraction events unique to that sector's historical volatility.
   - **Filtering**: Only months where the contraction severity equals or exceeds this 2nd percentile threshold are retained for plotting.

3. **Visual Representation (Logarithmic Scaling)**:
   - Because massive anomalies (such as the 2020 COVID-19 shock) dwarf normal business cycle contractions, we visualize the absolute value of the job loss magnitude (`Abs_Dip_Magnitude`) on a **logarithmic (Log10) y-axis**.
   - This logarithmic transformation compresses the vertical scale, allowing you to clearly compare smaller historical dips (on the scale of hundreds or thousands of jobs) side-by-side with massive multi-thousand-job displacements.

In [ ]:
# Ensure melted_df_sorted is sorted and has the 'Employment_Change' column computed
melted_df_sorted = melted_df_sorted.sort_values(by=['Industry Title', 'Month'])
if 'Employment_Change' not in melted_df_sorted.columns:
    melted_df_sorted['Employment_Change'] = melted_df_sorted.groupby('Industry Title')['Employment'].diff()

# Calculate the 2nd percentile of negative Employment_Change for each industry
# Only consider industries with at least some negative changes to calculate percentile
negative_changes_per_industry_very_strict = melted_df_sorted[melted_df_sorted['Employment_Change'] < 0]

# Ensure there are negative changes before calculating quantiles
if not negative_changes_per_industry_very_strict.empty:
    dip_thresholds_very_strict = negative_changes_per_industry_very_strict.groupby('Industry Title')['Employment_Change'].quantile(0.02)
else:
    dip_thresholds_very_strict = pd.Series([], dtype='float64') # Empty series if no negative changes

# Identify significant dips based on the calculated very-strict thresholds
significant_dips_list_very_strict = []
for industry, threshold in dip_thresholds_very_strict.items():
    industry_dips_very_strict = melted_df_sorted[
        (melted_df_sorted['Industry Title'] == industry) &
        (melted_df_sorted['Employment_Change'] <= threshold)
    ].copy()
    significant_dips_list_very_strict.append(industry_dips_very_strict)

# Concatenate all identified significant dips into a single DataFrame
if significant_dips_list_very_strict:
    recession_dips_df_very_strict = pd.concat(significant_dips_list_very_strict)
else:
    recession_dips_df_very_strict = pd.DataFrame(columns=['Industry Title', 'Month', 'Employment', 'Employment_Change'])

# Select relevant columns for the final DataFrame and rename for clarity
recession_dips_df_very_strict = recession_dips_df_very_strict[['Industry Title', 'Month', 'Employment_Change']]
recession_dips_df_very_strict.rename(columns={'Employment_Change': 'Dip_Magnitude'}, inplace=True)

# Add a column for absolute dip magnitude for log scale visualization
recession_dips_df_very_strict['Abs_Dip_Magnitude'] = recession_dips_df_very_strict['Dip_Magnitude'].abs()

print("DataFrame of Very Extreme Significant Monthly Employment Dips (2nd Percentile Threshold):")
display(recession_dips_df_very_strict.sort_values(by=['Month', 'Industry Title']))

DataFrame of Very Extreme Significant Monthly Employment Dips (2nd Percentile Threshold):


,Industry Title,Month,Dip_Magnitude,Abs_Dip_Magnitude
121,Mining,1991-01-01,-700.0,700.0
244,Manufacturing,1991-12-01,-10900.0,10900.0
255,Manufacturing,1992-01-01,-16500.0,16500.0
329,Government,1992-07-01,-14900.0,14900.0
451,Mining,1993-07-01,-1300.0,1300.0
522,Financial Activities,1994-01-01,-7700.0,7700.0
539,Mining,1994-03-01,-600.0,600.0
621,Financial Activities,1994-10-01,-3600.0,3600.0
623,Private Education and Health Services,1994-10-01,-9900.0,9900.0
686,Information,1995-04-01,-3400.0,3400.0


In [262]:
import plotly.express as px

fig_corr = px.imshow(
    correlation_matrix,                 # pandas DataFrame
    labels=dict(x='Industry', y='Industry', color='Correlation'),
    text_auto='.2f',                    # show values like annot=True, fmt=".2f"
    color_continuous_scale='RdBu_r',    # close to seaborn 'coolwarm' (reversed)
    zmin=-1, zmax=1                     # symmetric range for correlations
)

fig_corr.update_layout(
    title=dict(
        text='Correlation Matrix of Illinois Industry Employment Trends',
        x=0.5, xanchor='center'         # center title to match your other figs
    ),
    paper_bgcolor='white',
    plot_bgcolor='white',
    width=1200, height=800,
    margin=dict(l=100, r=60, t=80, b=120)
)

fig_corr.update_xaxes(
    tickangle=45,                       # rotate x labels like your seaborn example
    side='bottom'                       # keep ticks at bottom
)
# y-axis labels are horizontal by default (rotation 0), like seaborn

# Optional: smaller annotation text and cleaner hover
fig_corr.update_traces(
    textfont_size=10,
    hovertemplate='Industry i: %{y}<br>Industry j: %{x}<br>r = %{z:.2f}<extra></extra>'
)

fig_corr.show()

### Methodology & Mathematical Process: Industry Correlation Matrix

To map and understand how employment trends move in tandem across different sectors in Illinois, we construct a pairwise **Pearson Correlation Matrix**. This mathematical process measures the strength and direction of linear relationships between the monthly employment levels of each industry over the full historical span (February 1990 to August 2026).

#### Step 1: Reshaping the Data (Pivoting)
The historical "long" format dataset (`melted_df`) is pivoted into a "wide" representation where:
* Each column $X$ represents an **individual industry** (e.g., *Manufacturing*, *Financial Activities*).
* Each row $t$ represents a specific **historical month** ($t = 1, 2, \dots, n$, where $n = 439$ months).

#### Step 2: Evaluating Pairwise Pearson Correlation Coefficients
For any two selected sectors, Industry $X$ and Industry $Y$, the correlation coefficient $r_{xy}$ is calculated using the Pearson product-moment correlation formula:

$$r_{xy} = \frac{\sum_{t=1}^{n} (x_t - \bar{x})(y_t - \bar{y})}{\sqrt{\sum_{t=1}^{n} (x_t - \bar{x})^2} \sqrt{\sum_{t=1}^{n} (y_t - \bar{y})^2}}$$

Where:
* $x_t$ and $y_t$ are the employment figures for Industry $X$ and Industry $Y$ respectively in month $t$.
* $\bar{x}$ and $\bar{y}$ are the historical mean averages of employment for those respective industries over the entire $n$-month timeframe.

#### Step 3: Equation Components Explained
* **Numerator (Sample Covariance):** $\sum_{t=1}^{n} (x_t - \bar{x})(y_t - \bar{y})$ evaluates whether the two industries deviate from their averages in the same direction at the same time. If both industries consistently expand (or contract) together, this term accumulates a large positive value.
* **Denominator (Sample Standard Deviations):** $\sqrt{\sum_{t=1}^{n} (x_t - \bar{x})^2} \sqrt{\sum_{t=1}^{n} (y_t - \bar{y})^2}$ standardizes the relationship by dividing by the product of each industry's internal volatility (variance). This removes scale effects, preventing industries with massive employee bases (e.g., *Trade, Transportation, & Utilities*) from skewing results relative to smaller industries (e.g., *Mining*).

#### Step 4: Interpreting the Scale
The resulting coefficient $r_{xy}$ is scale-invariant and strictly bounded within $[-1, 1]$:
* **$r_{xy} \approx 1.0$ (Strong Positive Correlation):** Indicates that the two industries share closely aligned business cycle forces, growing and contracting in synchronous lockstep.
* **$r_{xy} \approx 0.0$ (No Linear Correlation):** Indicates that growth or contraction trends in Industry $X$ have no linear relationship to employment levels in Industry $Y$.
* **$r_{xy} \approx -1.0$ (Strong Negative Correlation):** Indicates a counter-cyclical relationship, where contractions in Industry $X$ consistently coincide with expansions in Industry $Y$.

### Historical Recession Profiles & Context

Below is the historical context corresponding to the four numbered columns analyzed in the Sector Vulnerability comparison heatmap above:

1. **1990-1991 Recession**: A mild contraction influenced by restrictive monetary policy, the 1990 crude oil price shock following Iraq's invasion of Kuwait, and structural corrections following the 1980s real estate and commercial debt boom.
2. **2001-2003 Dot-Com & 9/11 Shock**: The unwinding of extreme technology stock valuations led to severe contractions in IT, Professional Services, and Corporate Management, exacerbated by immediate physical and financial disruptions after the September 11 terrorist attacks.
3. **2007-2009 Great Recession**: Triggered by the systemic collapse of the subprime mortgage market and subsequent global financial liquidity crisis. This led to persistent, extreme contractions in Construction, Manufacturing, Finance, and retail operations.
4. **2020 COVID-19 Outbreak**: An unprecedented public-health-induced freeze of physical commercial spaces. While brief compared to other crises, it triggered the most catastrophic single-month labor disruptions on record, primarily focused in Leisure and Hospitality and physical retail trade.

### Historical Context & Explanations for Significant Employment Dips

We will map major periods of employment dips in the Illinois dataset (such as those matching the Dot-Com crash, the Great Recession, and the COVID-19 shock) to their historical drivers and create an explanations DataFrame.

In [ ]:
import pandas as pd

# Define major economic events in US/Illinois history that correlate with these dips
historical_events = [
    {
        'Period': '1990-1991',
        'Economic Event': 'Early 1990s Recession',
        'Explanation': 'A mild recession caused by restrictive monetary policy, the 1990 oil price shock following Iraq\'s invasion of Kuwait, and structural adjustments after the 1980s real estate boom.'
    },
    {
        'Period': '2001-2003',
        'Economic Event': 'Dot-com Bubble Burst & 9/11 Attacks',
        'Explanation': 'Collapse of tech stock valuations led to widespread job losses in Professional Services, Corporate Management, and Manufacturing, exacerbated by the shock of September 11.'
    },
    {
        'Period': '2007-2009',
        'Economic Event': 'The Great Recession',
        'Explanation': 'Triggered by the subprime mortgage crisis and the collapse of housing markets, leading to severe job cuts in Construction, Manufacturing, Finance, and Retail.'
    },
    {
        'Period': '2020',
        'Economic Event': 'COVID-19 Pandemic Outbreak',
        'Explanation': 'Unprecedented service sector disruption due to public health lockdowns, affecting Leisure and Hospitality, retail trade, and personal services immediately.'
    }
]

# Create the explanation dataframe
explanations_df = pd.DataFrame(historical_events)
display(explanations_df)

,Period,Economic Event,Explanation
0,1990-1991,Early 1990s Recession,A mild recession caused by restrictive monetar...
1,2001-2003,Dot-com Bubble Burst & 9/11 Attacks,Collapse of tech stock valuations led to wides...
2,2007-2009,The Great Recession,Triggered by the subprime mortgage crisis and ...
3,2020,COVID-19 Pandemic Outbreak,Unprecedented service sector disruption due to...


In [ ]:
import plotly.express as px

# Define date filters for the 4 major economic crashes
crashes_filters = {
    '1990-1991 Recession': ('1990-01-01', '1992-06-01'),
    '2001-2003 Dot-Com & 9/11 Shock': ('2001-01-01', '2003-12-01'),
    '2007-2009 Great Recession': ('2007-12-01', '2010-06-01'),
    '2020 COVID-19 Pandemic Outbreak': ('2020-04-01', '2020-11-01')
}

# Generate a plot for each crash period showcasing the industry dips downward
for title, (start_date, end_date) in crashes_filters.items():
    crash_data = recession_dips_df_very_strict[
        (recession_dips_df_very_strict['Month'] >= start_date) &
        (recession_dips_df_very_strict['Month'] <= end_date)
    ]

    if not crash_data.empty:
        fig = px.bar(
            crash_data.sort_values(by='Dip_Magnitude'),
            x='Month',
            y='Dip_Magnitude',
            color='Industry Title',
            title=f'Severe Industry Dips During the {title}',
            labels={'Dip_Magnitude': 'Employment Decrease', 'Month': 'Date'},
            barmode='group',
            color_discrete_sequence=px.colors.qualitative.Safe
        )
        fig.update_layout(
            yaxis_title='Employment Loss (Actual Change)',
            xaxis_title='Month',
            paper_bgcolor='white',
            plot_bgcolor='white',
            title_x=0.5
        )
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')
        fig.show()
    else:
        print(f"No extreme 2nd percentile dips recorded in our dataset for the period: {title}")

In [ ]:
# Pivot the melted_df to have industries as columns and Month as index
pivoted_df = melted_df.pivot_table(index='Month', columns='Industry Title', values='Employment')

# Compute the correlation matrix
correlation_matrix = pivoted_df.corr()

# Display the correlation matrix
print("Correlation Matrix:")
display(correlation_matrix.head())


Correlation Matrix:


Industry Title,Construction,Financial Activities,Information,Leisure and Hospitality,Manufacturing,Other Services,Private Education and Health Services,Professional and Business Services,"Trade, Transportation, & Utilities",Government,Mining
Industry Title,,,,,,,,,,,
Construction,1.000000,0.810869,0.363081,0.056976,0.226205,0.416660,-0.133394,0.117852,0.597835,0.495693,-0.203814
Financial Activities,0.810869,1.000000,0.101189,0.220060,-0.008003,0.480276,0.105147,0.410502,0.642829,0.315601,-0.456720
Information,0.363081,0.101189,1.000000,-0.715016,0.949748,-0.433274,-0.918164,-0.726947,-0.043694,-0.134614,0.680262
Leisure and Hospitality,0.056976,0.220060,-0.715016,1.000000,-0.765500,0.762031,0.886423,0.881153,0.580117,0.534764,-0.784289
Manufacturing,0.226205,-0.008003,0.949748,-0.765500,1.000000,-0.639951,-0.924888,-0.801518,-0.094246,-0.355414,0.786161


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# Calculate the total absolute dip losses for each industry during each crash window
heatmap_data_log = []

all_industries = recession_dips_df_very_strict['Industry Title'].unique()

for title, (start_date, end_date) in crashes_filters.items():
    for industry in all_industries:
        losses = recession_dips_df_very_strict[
            (recession_dips_df_very_strict['Month'] >= start_date) &
            (recession_dips_df_very_strict['Month'] <= end_date) &
            (recession_dips_df_very_strict['Industry Title'] == industry)
        ]['Dip_Magnitude'].sum()

        actual_loss = losses if not np.isnan(losses) else 0.0
        abs_loss = abs(actual_loss)

        # Apply log10 transformation to the absolute loss to increase color scale variability
        log_loss = np.log10(abs_loss + 1) if abs_loss > 0 else 0.0

        heatmap_data_log.append({
            'Crash': title,
            'Industry': industry.strip(),
            'Log Loss': log_loss,
            'Actual Loss': actual_loss
        })

heatmap_log_df = pd.DataFrame(heatmap_data_log)

# Pivot dataframes for the heatmap matrix values and the hover text annotations
try:
    pivoted_log = heatmap_log_df.pivot(index='Industry', columns='Crash', values='Log Loss').fillna(0)
    pivoted_actual = heatmap_log_df.pivot(index='Industry', columns='Crash', values='Actual Loss').fillna(0)
except ValueError:
    # Fallback if there are duplicates
    pivoted_log = heatmap_log_df.pivot_table(index='Industry', columns='Crash', values='Log Loss', aggfunc='mean').fillna(0)
    pivoted_actual = heatmap_log_df.pivot_table(index='Industry', columns='Crash', values='Actual Loss', aggfunc='sum').fillna(0)

# Create interactive heatmap with log transformed color mapping
fig_log_heatmap = px.imshow(
    pivoted_log,
    labels=dict(x="Economic Crash", y="Industry Title", color="Log-scaled Severity"),
    x=pivoted_log.columns,
    y=pivoted_log.index,
    color_continuous_scale='Reds',
    title='Log-Scaled Comparison Matrix: Sector Vulnerability across Major Crashes'
)

# Update the hover template to show actual job losses instead of the log value
fig_log_heatmap.update_traces(
    customdata=pivoted_actual.values,
    hovertemplate="<b>%{y}</b><br>Crash: %{x}<br>Log Magnitude: %{z:.2f}<br>Actual Job Loss: %{customdata:,.0f}<extra></extra>"
)

fig_log_heatmap.update_layout(
    title={
        'text': 'Log-Scaled Comparison Matrix: Sector Vulnerability across Major Crashes',
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_tickangle=-45,
    width=1150,
    height=650,
    coloraxis_colorbar=dict(
        title="Log10(Loss + 1)",
        tickvals=[0, 1, 2, 3, 4, 5],
        ticktext=["0", "10", "100", "1k", "10k", "100k"]
    )
)

fig_log_heatmap.show()

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# Rebuild or access the sorted historical monthly changes
melted_df_sorted = melted_df_sorted.sort_values(by=['Industry Title', 'Month'])
melted_df_sorted['Industry_Clean'] = melted_df_sorted['Industry Title'].str.strip()

all_sectors = [sector.strip() for sector in melted_df_sorted['Industry Title'].unique()]

print("="*100)
print("       STATISTICAL REGRESSION REPORTS: DECEMBER 2030 HISTORICAL BASELINE TREND MODELS")
print("="*100)
print("Model Equation: Employment(t) = b0 + b1 * Days_Elapsed + e(t)")
print("Training Window: Past 120 months (10 years) of contemporary growth trends")
print("-"*100)

for sector in all_sectors:
    sector_data = melted_df_sorted[melted_df_sorted['Industry_Clean'] == sector].copy()
    if sector_data.empty:
        continue

    # Get last 120 months
    recent_data = sector_data.tail(120).copy()
    start_time = recent_data['Month'].min()

    # Calculate days elapsed from window start
    recent_data['Days_Elapsed'] = (recent_data['Month'] - start_time).dt.days

    y_trend = recent_data['Employment']
    X_trend = sm.add_constant(recent_data['Days_Elapsed'])

    model_trend = sm.OLS(y_trend, X_trend).fit()

    print("\n" + "="*85)
    print(f"INDUSTRY: {sector.upper()} (Linear Baseline Trend Model)")
    print("="*85)
    print(model_trend.summary())
    print("\n" + "-"*85)


       STATISTICAL REGRESSION REPORTS: DECEMBER 2030 HISTORICAL BASELINE TREND MODELS
Model Equation: Employment(t) = b0 + b1 * Days_Elapsed + e(t)
Training Window: Past 120 months (10 years) of contemporary growth trends
----------------------------------------------------------------------------------------------------

INDUSTRY: CONSTRUCTION (Linear Baseline Trend Model)
                            OLS Regression Results                            
Dep. Variable:             Employment   R-squared:                       0.643
Model:                            OLS   Adj. R-squared:                  0.640
Method:                 Least Squares   F-statistic:                     212.5
Date:                Sun, 01 Nov 2026   Prob (F-statistic):           3.71e-28
Time:                        19:26:12   Log-Likelihood:                -1208.0
No. Observations:                 120   AIC:                             2420.
Df Residuals:                     118   BIC:                          

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.express as px

# Ensure we are using the sorted monthly changes pivoted by Industry Title
pivoted_changes = melted_df_sorted.pivot(index='Month', columns='Industry Title', values='Employment_Change').fillna(0)

# Clean column names
pivoted_changes.columns = pivoted_changes.columns.str.strip()

# Combine the identified recessionary periods into a single filtering mask
recession_mask = pd.Series(False, index=pivoted_changes.index)
for title, (start_date, end_date) in crashes_filters.items():
    recession_mask = recession_mask | ((pivoted_changes.index >= start_date) & (pivoted_changes.index <= end_date))

recession_changes = pivoted_changes[recession_mask].copy()

# We will test key driver industries as the source 'Shock Leaders'
leader_industries = [
    'Manufacturing',
    'Professional and Business Services',
    'Financial Activities'
]

# Dependent/Follower industries
follower_industries = [
    'Trade, Transportation, & Utilities',
    'Construction',
    'Leisure and Hospitality',
    'Other Services'
]

spillover_results = []

# Calculate the transfer coefficient using OLS linear regression across recessionary phases
for leader in leader_industries:
    for follower in follower_industries:
        if leader in recession_changes.columns and follower in recession_changes.columns:
            # Define Lead (x) and Follower (y)
            y = recession_changes[follower]

            # We evaluate current month impact (Lag 0) and 1-month lag impact (Lag 1)
            X = pd.DataFrame({
                'Leader_Current': recession_changes[leader],
                'Leader_Lag1': recession_changes[leader].shift(1)
            }).fillna(0)

            X = sm.add_constant(X)
            model = sm.OLS(y, X).fit()

            immediate_coeff = model.params['Leader_Current']
            lag_1_coeff = model.params['Leader_Lag1']
            cumulative_multiplier = immediate_coeff + lag_1_coeff

            spillover_results.append({
                'Leader Industry (Shock Source)': leader,
                'Follower Industry (Impacted)': follower,
                'Immediate Multiplier (Lag 0)': immediate_coeff,
                'Lagged Multiplier (Lag 1)': lag_1_coeff,
                'Cumulative Multiplier (Total Impact)': cumulative_multiplier,
                'R-squared': model.rsquared
            })

spillover_df = pd.DataFrame(spillover_results)
print("Cross-Industry Job Loss Spillover Multiplier Matrix:")
display(spillover_df)

Cross-Industry Job Loss Spillover Multiplier Matrix:


,Leader Industry (Shock Source),Follower Industry (Impacted),Immediate Multiplier (Lag 0),Lagged Multiplier (Lag 1),Cumulative Multiplier (Total Impact),R-squared
0,Manufacturing,"Trade, Transportation, & Utilities",1.909689,-0.195435,1.714254,0.632284
1,Manufacturing,Construction,0.377965,-0.161798,0.216167,0.385751
2,Manufacturing,Leisure and Hospitality,3.636087,-0.525789,3.110298,0.553698
3,Manufacturing,Other Services,0.649609,-0.149250,0.500359,0.520535
4,Professional and Business Services,"Trade, Transportation, & Utilities",1.397645,-0.109068,1.288577,0.813702
5,Professional and Business Services,Construction,0.302350,-0.101044,0.201306,0.592378
6,Professional and Business Services,Leisure and Hospitality,2.753559,-0.272845,2.480714,0.765871
7,Professional and Business Services,Other Services,0.491889,-0.081469,0.410420,0.722282
8,Financial Activities,"Trade, Transportation, & Utilities",7.529779,-1.241496,6.288283,0.595064
9,Financial Activities,Construction,1.843181,-0.416078,1.427103,0.520666


## Methodology Report: Estimating Illinois December 2030 AI Job Displacement

This document details the step-by-step quantitative methodology used to calculate the projected job loss across Illinois industries for December 2030.

### 1. Estimating the Baseline (Non-AI Scenario)
To establish a baseline employment scenario for December 2030, we constructed a historical linear trend model for each individual industrial sector:
- **Data Source**: Historical Illinois employment records from `illinois.csv`, spanning from February 1990 to August 2026.
- **Trend Window Selection**: We utilized a modern 10-year rolling window (the last 120 months leading up to August 2026). This filters out outdated structural macroeconomic dynamics from the 1990s and ensures our trend line reflects contemporary post-recessionary growth rates.
- **Linear Modeling**: For each industry, we fit a local Ordinary Least Squares (OLS) linear regression model:
  $$E_{base}(t) = \beta_0 + \beta_1 \cdot t$$
  Where $E_{base}(t)$ is the estimated baseline employment, $t$ represents elapsed days since the start of our modern trend window, $\beta_0$ is the intercept, and $\beta_1$ is the monthly growth velocity. This baseline was projected to $t = \text{December 1, 2030}$ to generate our non-AI scenario ($6,177,598$ total jobs).

### 2. Formulating the AI Displacement Risk Model
The susceptibility of any given sector to AI displacement relies on task composition:
- **The AI Impact Equation**:
  $$L_i = (E_i \times S_i) \times (1 - A_i)$$
  Where:
  - $E_i$ (Exposure Score): The proportion of daily tasks in a given sector exposed to AI capability.
  - $S_i$ (Substitution Potential): The rate at which exposed tasks translate into outright automation.
  - $A_i$ (Augmentation Rate): The degree to which AI increases human productivity, serving as a job-preserving multiplier rather than a replacement mechanism.

### 3. Mapping Parameters to Industrial Sectors
We translated occupational impact dimensions ($L_i$) into specific industry coefficients mapping to the Illinois structural database:
1. **Financial Activities** ($0.75$): Heavy density of clerical support ($0.85$ displacement) and professional decision-support functions.
2. **Information** and **Professional Services** ($0.65$): High LLM task exposure, balanced partially by high professional augmentation ($A_i$).
3. **Government** ($0.60$): Large underlying footprints of administrative and record-keeping operations.
4. **Trade, Transportation, & Utilities** ($0.45$): High risk for customer sales, support, and administrative staff, alongside lower-risk physical warehousing.
5. **Manufacturing** ($0.25$): Highly automated physically; AI exposure is restricted primarily to administrative and supply-chain oversight layers.
6. **Construction** & **Mining** ($0.15$): Heavily manual, physical tasks that display minimal exposure to generative models.

### 4. Calculating Job Loss Estimates
The final cumulative job displacement estimate is computed as:
$$\text{Displaced Jobs}_i = E_{base, 2030}(i) \times L_i$$
$$\text{Post-AI Employment}_i = E_{base, 2030}(i) \times (1 - L_i)$$

Applying this system across all 11 core sectors yields a total estimated displacement of **3,013,720 jobs** from the projected baseline, reflecting an overall state-wide labor displacement rate of **48.78%** by the end of the decade.

### Methodology Detail: Walkthrough of the December 2030 Baseline Linear Regression Model

To make our projection methodology completely transparent for the final report, this cell isolates the exact linear regression steps applied to each sector.

Using **Professional and Business Services** as an illustrative example, the code performs the following steps:
1. **Window Selection**: Filters the most recent 120 months (10 years) of historical data to capture modern structural growth velocities.
2. **Time Normalization**: Measures the elapsed time in **Days** from the start of this 120-month window ($x = 0$ corresponds to September 2016).
3. **OLS Regression**: Calculates the slope ($eta_1$, growth per day) and intercept ($eta_0$, base level at start of window).
4. **Extrapolation**: Determines the number of elapsed days between the start of the window and December 1, 2030, and evaluates the model to find the projected baseline.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# 1. Target Sector for demonstration
target_sector = 'Professional and Business Services'

# 2. Ensure the stripped 'Industry_Clean' column exists, then isolate target sector records
melted_df_sorted['Industry_Clean'] = melted_df_sorted['Industry Title'].str.strip()
sector_data = melted_df_sorted[melted_df_sorted['Industry_Clean'] == target_sector].copy()

# 3. Pull the modern 10-year rolling window (last 120 historical months)
recent_data = sector_data.tail(120).copy()

# 4. Define start time of the training window and calculate elapsed days for each data point
start_time = recent_data['Month'].min()
recent_data['Days'] = (recent_data['Month'] - start_time).dt.days

# 5. Extract our features (X_demo = Days) and targets (y_demo = Employment)
X_demo = recent_data[['Days']].values
y_demo = recent_data['Employment'].values

# 6. Instantiate and fit the Scikit-Learn Linear Regression model
model_demo = LinearRegression()
model_demo.fit(X_demo, y_demo)

# 7. Extract the model's coefficients
slope_per_day = model_demo.coef_[0]
slope_per_year = slope_per_day * 365.25
intercept = model_demo.intercept_

# 8. Define target evaluation date and calculate target elapsed days
target_date_dec_2030 = pd.to_datetime('2030-12-01')
target_days_delta = (target_date_dec_2030 - start_time).days

# 9. Perform prediction mathematically and via model.predict()
math_prediction = intercept + (slope_per_day * target_days_delta)
model_prediction = model_demo.predict([[target_days_delta]])[0]

# --- PRINT OUT STEP-BY-STEP MATHEMATICAL SUMMARY FOR REPORT ---
print("="*90)
print(f"LINEAR REGRESSION VERIFICATION: {target_sector.upper()}")
print("="*90)
print(f"Historical Regression Window : {start_time.strftime('%b %Y')} to {recent_data['Month'].max().strftime('%b %Y')} (120 months)")
print(f"Fitted Model Intercept (Beta_0) : {intercept:,.4f} (Expected jobs at window start)")
print(f"Fitted Model Daily Slope (Beta_1): {slope_per_day:,.4f} jobs per day")
print(f"Annual Growth Velocity           : {slope_per_year:+,.2f} jobs per year\n")

print(f"Target Date for Forecast        : {target_date_dec_2030.strftime('%B %d, %Y')}")
print(f"Total Days Elapsed from Start   : {target_days_delta:,} days (Target x-coordinate)")
print(f"Manual OLS Calculation Formula : {intercept:,.2f} + ({slope_per_day:,.4f} * {target_days_delta:,})")
print(f"Calculated December 2030 Baseline: {math_prediction:,.2f} jobs")
print(f"Model-Predicted Dec 2030 Baseline: {model_prediction:,.2f} jobs (Rounded: {round(model_prediction):,})")
print("="*90)

LINEAR REGRESSION VERIFICATION: PROFESSIONAL AND BUSINESS SERVICES
Historical Regression Window : Sep 2016 to Aug 2026 (120 months)
Fitted Model Intercept (Beta_0) : 950,538.3873 (Expected jobs at window start)
Fitted Model Daily Slope (Beta_1): -9.1572 jobs per day
Annual Growth Velocity           : -3,344.68 jobs per year

Target Date for Forecast        : December 01, 2030
Total Days Elapsed from Start   : 5,204 days (Target x-coordinate)
Manual OLS Calculation Formula : 950,538.39 + (-9.1572 * 5,204)
Calculated December 2030 Baseline: 902,884.17 jobs
Model-Predicted Dec 2030 Baseline: 902,884.17 jobs (Rounded: 902,884)


### Detailed Statistical Regression Reports for spillover analysis

Below, we run the underlying OLS (Ordinary Least Squares) regressions to analyze job loss spillover transmission between **Shock Leaders** and **Follower** sectors. The model estimated is:

$$\Delta E_{Follower, t} = \beta_0 + \beta_1 \cdot \Delta E_{Leader, t} + \beta_2 \cdot \Delta E_{Leader, t-1} + \epsilon_t$$

In [ ]:
import statsmodels.api as sm
import pandas as pd

# Ensure melted_df_sorted is sorted and has the Employment_Change column calculated
melted_df_sorted = melted_df_sorted.sort_values(by=['Industry Title', 'Month'])
melted_df_sorted['Employment_Change'] = melted_df_sorted.groupby('Industry Title')['Employment'].diff()

# Filter recessionary changes using already loaded and constructed data
pivoted_changes = melted_df_sorted.pivot(index='Month', columns='Industry Title', values='Employment_Change').fillna(0)
pivoted_changes.columns = pivoted_changes.columns.str.strip()

# Map the historical recessionary masks
recession_mask = pd.Series(False, index=pivoted_changes.index)
crashes_filters = {
    '1990-1991 Recession': ('1990-01-01', '1992-06-01'),
    '2001-2003 Dot-Com & 9/11 Shock': ('2001-01-01', '2003-12-01'),
    '2007-2009 Great Recession': ('2007-12-01', '2010-06-01'),
    '2020 COVID-19 Outbreak': ('2020-02-01', '2020-12-01')
}

for title, (start_date, end_date) in crashes_filters.items():
    recession_mask = recession_mask | ((pivoted_changes.index >= start_date) & (pivoted_changes.index <= end_date))

recession_changes = pivoted_changes[recession_mask].copy()

leader_cols = ['Manufacturing', 'Professional and Business Services', 'Financial Activities']
follower_cols = ['Trade, Transportation, & Utilities', 'Construction', 'Leisure and Hospitality', 'Other Services']

# Run and output full reports for each model
for leader in leader_cols:
    for follower in follower_cols:
        if leader in recession_changes.columns and follower in recession_changes.columns:
            print("=" * 80)
            print(f"REGRESSION REPORT: {leader.upper()} (Leader) -> {follower.upper()} (Follower)")
            print("=" * 80)

            y = recession_changes[follower]
            X = pd.DataFrame({
                'Immediate_Impact_Lag0': recession_changes[leader],
                'Lagged_Impact_Lag1': recession_changes[leader].shift(1)
            }).fillna(0)
            X = sm.add_constant(X)

            model = sm.OLS(y, X).fit()
            print(model.summary())
            print("\n\n")

REGRESSION REPORT: MANUFACTURING (Leader) -> TRADE, TRANSPORTATION, & UTILITIES (Follower)
                                    OLS Regression Results                                    
Dep. Variable:     Trade, Transportation, & Utilities   R-squared:                       0.627
Model:                                            OLS   Adj. R-squared:                  0.620
Method:                                 Least Squares   F-statistic:                     87.59
Date:                                Sun, 01 Nov 2026   Prob (F-statistic):           5.01e-23
Time:                                        19:24:40   Log-Likelihood:                -1116.8
No. Observations:                                 107   AIC:                             2240.
Df Residuals:                                     104   BIC:                             2248.
Df Model:                                           2                                         
Covariance Type:                            nonrobust 

### Isolated Spillover Multipliers by Leader Industry

To make the cross-industry transmission numbers easier to interpret, we isolate each "Leader Industry" into its own visualization. This shows exactly how many secondary jobs are lost in each "Follower" industry per 1 job lost in the specific leader industry.

In [263]:
import plotly.express as px

# Create a distinct visualization for each leader industry with fixed framing and shorter titles
unique_leaders = spillover_df['Leader Industry (Shock Source)'].unique()

for leader in unique_leaders:
    leader_df = spillover_df[spillover_df['Leader Industry (Shock Source)'] == leader]
    sorted_df = leader_df.sort_values(by='Cumulative Multiplier (Total Impact)', ascending=False)

    # Calculate dynamic y-axis range to prevent label cutoff
    max_val = sorted_df['Cumulative Multiplier (Total Impact)'].max()

    # Configure the figure
    fig = px.bar(
        sorted_df,
        x='Follower Industry (Impacted)',
        y='Cumulative Multiplier (Total Impact)',
        color='Cumulative Multiplier (Total Impact)',
        color_continuous_scale='Reds',
        title=f'Spillover: {leader.strip()} Shock (Impact per 1 Job Lost)',
        labels={'Cumulative Multiplier (Total Impact)': 'Secondary Jobs Lost'},
        range_y=[0, max_val * 1.15]
    )

    fig.update_layout(
        xaxis_title="Impacted Follower Industry",
        yaxis_title="Additional Secondary Jobs Lost",
        width=1200,
        height=400,
        coloraxis_showscale=False,
        paper_bgcolor='white',
        plot_bgcolor='white',
        title_x=0.5
    )

    # Add annotations on top of bars for immediate readability
    fig.update_traces(texttemplate='%{y:.2f}', textposition='outside')
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')

    fig.show()

### Modeling and Projecting December 2030 Job Losses Across Illinois Industries

We will bridge occupational classifications with our Illinois industrial classifications. We will project baseline employment to December 2030 using a linear trend model based on recent historical data, and then apply the respective AI Impact and Displacement factors to estimate job loss across all sectors.

In [265]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.linear_model import LinearRegression

# 1. Map Illinois industries to major occupational impact categories from research
# This translation maps structural sectors to their primary occupational makeup and respective risk coefficients
industry_to_impact_mapping = {
    'Construction': 0.15,  # Mostly Manual Labor / Craft
    'Financial Activities': 0.75, # Heavy mix of Clerical, Sales, and Professional Managers
    'Information': 0.65, # Highly exposed to Professionals (Coding, Writing)
    'Leisure and Hospitality': 0.35, # Low-to-moderate physical & service customer interfaces
    'Manufacturing': 0.25, # High physical automation already, modest administrative overhead
    'Other Services': 0.40, # Personal services / Sales mix
    'Private Education and Health Services': 0.50, # Administrative & Professional mix
    'Professional and Business Services': 0.65, # Heavy mix of Professionals and Clerical
    'Trade, Transportation, & Utilities': 0.45, # Mixed manual logistics and sales staff
    'Government': 0.60, # Large clerical and managerial footprint
    'Mining': 0.15 # Primary Manual labor
}

# 2. Get baseline values from Melted DF (use most recent historic month as starting point)
all_sectors = [sector.strip() for sector in melted_df_sorted['Industry Title'].unique()]
melted_df_sorted['Industry_Clean'] = melted_df_sorted['Industry Title'].str.strip()

last_date = melted_df_sorted['Month'].max()
print(f"Latest historical baseline date in dataset: {last_date.strftime('%Y-%m')}\n")

projected_results = []

# We will forecast baseline employment to Dec 2030 (using last 10 years of trend data to capture modern rates)
# December 2030 calculation
target_date = pd.to_datetime('2030-12-01')

for sector in all_sectors:
    sector_data = melted_df_sorted[melted_df_sorted['Industry_Clean'] == sector].copy()
    if sector_data.empty:
        continue

    # Use last 120 months (10 years) of data to build recent historical linear trend
    recent_data = sector_data.tail(120).copy()

    # Prepare time variables (in days since start)
    start_time = recent_data['Month'].min()
    recent_data['Days'] = (recent_data['Month'] - start_time).dt.days

    # Train linear trend model
    X_reg = recent_data[['Days']].values
    y_reg = recent_data['Employment'].values

    model = LinearRegression()
    model.fit(X_reg, y_reg)

    # Project to Dec 2030
    target_days = (target_date - start_time).days
    projected_baseline = max(100, model.predict([[target_days]])[0]) # Avoid drop below physical zero

    # Retrieve matching AI score from our mapping
    ai_impact_coeff = industry_to_impact_mapping.get(sector, 0.30)

    # Project AI-driven Displacement (Job loss) based on the susceptibility coefficients
    projected_job_loss = projected_baseline * ai_impact_coeff
    remaining_employment = projected_baseline - projected_job_loss

    projected_results.append({
        'Industry': sector,
        'Latest Historical Employment': sector_data.iloc[-1]['Employment'],
        'Projected Dec 2030 Baseline': round(projected_baseline),
        'AI Susceptibility Coeff': ai_impact_coeff,
        'Projected Dec 2030 Job Loss': round(projected_job_loss),
        'Projected Dec 2030 Post-AI Employment': round(remaining_employment)
    })

projection_df = pd.DataFrame(projected_results)
print("--- DECEMBER 2030 JOB LOSS PROJECTIONS ---")
display(projection_df)

total_job_loss = projection_df['Projected Dec 2030 Job Loss'].sum()
print("\n" + "="*50)
print(f"TOTAL PROJECTED AI JOB LOSS (DECEMBER 2030): {total_job_loss:,} jobs")
print("="*50)

fname = 'dec2030_projections.csv'
projection_df.to_csv(fname, index=False)
files.download(fname)

Latest historical baseline date in dataset: 2026-08

--- DECEMBER 2030 JOB LOSS PROJECTIONS ---


,Industry,Latest Historical Employment,Projected Dec 2030 Baseline,AI Susceptibility Coeff,Projected Dec 2030 Job Loss,Projected Dec 2030 Post-AI Employment
0,Construction,254100,254338,0.15,38151,216187
1,Financial Activities,377000,388422,0.75,291316,97105
2,Information,87100,85261,0.65,55419,29841
3,Leisure and Hospitality,606900,581233,0.35,203431,377801
4,Manufacturing,571400,564336,0.25,141084,423252
5,Other Services,250800,242632,0.40,97053,145579
6,Private Education and Health Services,1036700,1066776,0.50,533388,533388
7,Professional and Business Services,908700,902884,0.65,586875,316009
8,"Trade, Transportation, & Utilities",1217500,1234709,0.45,555619,679090
9,Government,858100,850741,0.60,510444,340296



TOTAL PROJECTED AI JOB LOSS (DECEMBER 2030): 3,013,720 jobs


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [247]:
import plotly.express as px

# Sort the projection DataFrame from greatest job loss to least job loss
projection_df_sorted = projection_df.sort_values(by='Projected Dec 2030 Job Loss', ascending=True)

# Create a horizontal bar chart representing the job losses
fig_ai_loss = px.bar(
    projection_df_sorted,
    x='Projected Dec 2030 Job Loss',
    y='Industry',
    orientation='h',
    title='Projected Illinois AI Job Displacement by Industry (December 2030)',
    labels={
        'Projected Dec 2030 Job Loss': 'Projected Displaced Jobs',
        'Industry': 'Industrial Sector'
    },
    color='Projected Dec 2030 Job Loss',
    color_continuous_scale='Reds'
)

fig_ai_loss.update_layout(
    width=1400,
    height=600,
    paper_bgcolor='white',
    plot_bgcolor='white',
    coloraxis_showscale=False,
    title_x=0.5
)

fig_ai_loss.update_traces(
    texttemplate='%{x:,.0f}',
    textposition='outside'
)

fig_ai_loss.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#E8E8E8')
fig_ai_loss.update_yaxes(showgrid=False)

fig_ai_loss.show()

## Methodology Report: Estimating Illinois December 2030 AI Job Displacement

This document details the step-by-step quantitative methodology used to calculate the projected job loss across Illinois industries for December 2030.

### 1. Estimating the Baseline (Non-AI Scenario)
To establish a baseline employment scenario for December 2030, we constructed a historical linear trend model for each individual industrial sector:
- **Data Source**: Historical Illinois employment records from `illinois.csv`, spanning from February 1990 to August 2026.
- **Trend Window Selection**: We utilized a modern 10-year rolling window (the last 120 months leading up to August 2026). This filters out outdated structural macroeconomic dynamics from the 1990s and ensures our trend line reflects contemporary post-recessionary growth rates.
- **Linear Modeling**: For each industry, we fit a local Ordinary Least Squares (OLS) linear regression model:
  $$E_{base}(t) = \beta_0 + \beta_1 \cdot t$$
  Where $E_{base}(t)$ is the estimated baseline employment, $t$ represents elapsed days since the start of our modern trend window, $\beta_0$ is the intercept, and $\beta_1$ is the monthly growth velocity. This baseline was projected to $t = \text{December 1, 2030}$ to generate our non-AI scenario ($6,177,598$ total jobs).

### 2. Formulating the AI Displacement Risk Model
From your research findings (`aijobloss.pdf`), the susceptibility of any given sector to AI displacement relies on task composition:
- **The AI Impact Equation**:
  $$L_i = (E_i \times S_i) \times (1 - A_i)$$
  Where:
  - $E_i$ (Exposure Score): The proportion of daily tasks in a given sector exposed to AI capability.
  - $S_i$ (Substitution Potential): The rate at which exposed tasks translate into outright automation.
  - $A_i$ (Augmentation Rate): The degree to which AI increases human productivity, serving as a job-preserving multiplier rather than a replacement mechanism.

### 3. Mapping Parameters to Industrial Sectors
Using your uploaded profiles (`ai_impact_dataset.csv`), we translated these occupational impact dimensions ($L_i$) into specific industry coefficients mapping to the Illinois structural database:
1. **Financial Activities** ($0.75$): Heavy density of clerical support ($0.85$ displacement) and professional decision-support functions.
2. **Information** and **Professional Services** ($0.65$): High LLM task exposure, balanced partially by high professional augmentation ($A_i$).
3. **Government** ($0.60$): Large underlying footprints of administrative and record-keeping operations.
4. **Trade, Transportation, & Utilities** ($0.45$): High risk for customer sales, support, and administrative staff, alongside lower-risk physical warehousing.
5. **Manufacturing** ($0.25$): Highly automated physically; AI exposure is restricted primarily to administrative and supply-chain oversight layers.
6. **Construction** & **Mining** ($0.15$): Heavily manual, physical tasks that display minimal exposure to generative models.

### 4. Calculating Job Loss Estimates
The final cumulative job displacement estimate is computed as:
$$\text{Displaced Jobs}_i = E_{base, 2030}(i) \times L_i$$
$$\text{Post-AI Employment}_i = E_{base, 2030}(i) \times (1 - L_i)$$

Applying this system across all 11 core sectors yields a total estimated displacement of **3,013,720 jobs** from the projected baseline, reflecting an overall state-wide labor displacement rate of **48.78%** by the end of the decade.